In [5]:
import os
from pathlib import Path

ROOT_DIR_BRAINTREEBANK = Path(r"C:\Users\simon\PyCharmMiscProject\neuroprobe-dev\braintreebank")
os.environ["ROOT_DIR_BRAINTREEBANK"] = str(ROOT_DIR_BRAINTREEBANK)

assert ROOT_DIR_BRAINTREEBANK.exists(), ROOT_DIR_BRAINTREEBANK

In [7]:
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch

from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score, accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

import neuroprobe
from neuroprobe import (
    BrainTreebankSubject,
    BrainTreebankSubjectTrialBenchmarkDataset,
    generate_splits_cross_session,
    generate_splits_cross_subject,
    generate_splits_within_session,
)

In [9]:
import neuroprobe.config as neuroprobe_config

print("BrainTreebank root:", ROOT_DIR_BRAINTREEBANK)
print("Neuroprobe ROOT_DIR:", neuroprobe_config.ROOT_DIR)
print("Sampling rate:", neuroprobe_config.SAMPLING_RATE, "Hz")

# Subject setup
subject_id = 1
trial_id = 1
coordinates_type = "mni"  # "mni", "mni305", "cortical", "lpi"

subject = BrainTreebankSubject(
    subject_id=subject_id,
    allow_corrupted=False,
    cache=True,
    dtype=torch.float32,
    coordinates_type=coordinates_type,
)
print("Loaded subject", subject_id)
print("First 10 electrode labels:", subject.electrode_labels[:10])
print("First 10 electrode MNI coordinates:")
print(subject.get_electrode_coordinates()[:10])

# Benchmark dataset setup
eval_name = "volume"
output_indices = False
start_neural_data_before_word_onset = 0
end_neural_data_after_word_onset = neuroprobe_config.SAMPLING_RATE * 1  # 1 second

benchmark_dataset = BrainTreebankSubjectTrialBenchmarkDataset(
    subject,
    trial_id,
    dtype=torch.float32,
    eval_name=eval_name,
    output_indices=output_indices,
    start_neural_data_before_word_onset=start_neural_data_before_word_onset,
    end_neural_data_after_word_onset=end_neural_data_after_word_onset,
    lite=True,
)

data_electrode_labels = benchmark_dataset.electrode_labels
data_electrode_coordinates = benchmark_dataset.electrode_coordinates

print("Dataset type:", type(benchmark_dataset))
print("Dataset length:", len(benchmark_dataset))

first_item = benchmark_dataset[0]
print("First item type:", type(first_item))
print("First item:", first_item)

if isinstance(first_item, dict):
    print("First item data shape:", first_item["data"].shape)
    print("First item label:", first_item["label"])
else:
    print("First item shape:", first_item[0].shape)
    print("First item label:", first_item[1])

print("Number of electrodes in dataset:", len(data_electrode_labels))

BrainTreebank root: C:\Users\simon\PyCharmMiscProject\neuroprobe-dev\braintreebank
Neuroprobe ROOT_DIR: C:\Users\simon\PyCharmMiscProject\neuroprobe-dev\braintreebank
Sampling rate: 2048 Hz
Loaded subject 1
First 10 electrode labels: ['F3aOFa2', 'F3aOFa3', 'F3aOFa4', 'F3aOFa7', 'F3aOFa8', 'F3aOFa9', 'F3aOFa10', 'F3aOFa11', 'F3aOFa12', 'F3aOFa13']
First 10 electrode MNI coordinates:
tensor([[  8.0828,  44.3820, -15.1744],
        [ 12.4152,  43.6956, -14.8598],
        [ 15.7043,  42.0086, -13.4021],
        [ 27.5911,  38.1577,  -9.0090],
        [ 30.7824,  37.5062,  -7.6428],
        [ 35.1301,  35.9065,  -6.1294],
        [ 38.4192,  34.2195,  -4.6717],
        [ 42.6692,  33.6553,  -3.2497],
        [ 45.9583,  31.9683,  -1.7920],
        [ 50.2907,  31.2818,  -1.4774]])
Dataset type: <class 'neuroprobe.datasets.BrainTreebankSubjectTrialBenchmarkDataset'>
Dataset length: 3500
First item type: <class 'dict'>
First item: {'data': tensor([[ 32.9645,  27.3818,  20.4699,  ...,   2.1267,